# 06 — Final değerlendirme (test seti)

**Çalışma zamanı: T4 GPU yeterli.** Eğitim yok. Toplam ~20 dk, çoğu dataset'i açmak.

**Kural:** model seçimi bitti (E05, val'e göre). Test seti burada **ilk ve tek kez** açılıyor; sonuca bakıp model ya da eşik değiştirilmez, yoksa test de val'e dönüşür.

**Akış**
1. **Güven eşiği val'de seçilir.** Her sınıf için: val'de kare bazında yanlış alarm ≤ `FA_MAX` olan en düşük eşik (en çok yakalayan). Test'e bakmadan sabitlenir.
2. **Test, standart ölçüler:** Ultralytics mAP50 / mAP50-95. Tüm test, FASDD test ve **Heinola** (modelin hiç görmediği yangın, sadece duman) ayrı ayrı.
3. **Test, pratik ölçüler:** kare bazında yakalama ve yanlış alarm, sahne ortalaması, piksel kapsama.
4. **Hata örnekleri:** kaçırılan ateşler, yanlış alarmlar, Heinola'da kaçırılan duman.
5. **Rapor:** `reports/final_eval.md`. README ve portföy için.

Karşılaştırma için baseline (E01) de aynı ölçülerle test edilir. Bu seçim değil, ilerlemeyi göstermek için.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!pip -q install ultralytics==8.4.183

In [ ]:
import os, json, subprocess, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
from PIL import Image
import torch
from ultralytics import YOLO

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
RUNS   = ROOT / 'runs'

# --- Ayarlar -------------------------------------------------------------------
FINAL   = 'E05_y11s_640_dfire'   # val'e göre seçilen model
BASE    = 'E01_y11s_640'         # karşılaştırma için baseline
DSET    = 'firesmoke_v2'         # val ve test her sürümde aynı
FA_MAX  = 1.0                    # val'de izin verilen kare bazında yanlış alarm (%)
CONFS   = np.round(np.arange(0.05, 0.71, 0.05), 2)
SEED    = 0
# ------------------------------------------------------------------------------

DS_ZIP = ROOT / 'datasets' / f'{DSET}.zip'
DATA   = Path('/content') / DSET
OUTD   = REP / 'final_eval'
OUTD.mkdir(parents=True, exist_ok=True)
NAMES  = {0: 'fire', 1: 'smoke'}
WORKERS = min(8, os.cpu_count() or 2)
IMGSZ = {e: int(yaml.safe_load((RUNS / e / 'args.yaml').read_text())['imgsz']) for e in (FINAL, BASE)}
for e in (FINAL, BASE):
    assert (RUNS / e / 'weights' / 'best.pt').exists(), e
print(IMGSZ)

## 1. Dataset'i aç

In [ ]:
t0 = time.time()
if not (DATA / 'data.yaml').exists():
    subprocess.run(['unzip', '-q', '-o', str(DS_ZIP), '-d', str(DATA.parent)], check=True)
man = pd.read_csv(DATA / 'manifest.csv')
val  = man[man.split == 'val'].reset_index(drop=True)
test = man[man.split == 'test'].reset_index(drop=True)
print(f'val {len(val)} · test {len(test)}  ({time.time() - t0:.0f} sn)')
display(test.groupby(['source', 'cat']).size().unstack(fill_value=0))

## 2. Tahminler (val + test, iki model)

Her görüntü için tahminler bir kez alınır (conf ≥ 0.05); eşik taraması ve tüm ölçümler bunları kullanır.

In [ ]:
def read_gt(df):
    gt = {}
    for uid, f in zip(df.uid, df.file):
        p = Path(str(DATA / f.replace('images/', 'labels/', 1).rsplit('.', 1)[0]) + '.txt')
        a = np.loadtxt(p, ndmin=2) if p.exists() and p.stat().st_size else np.zeros((0, 5))
        if len(a):
            c, x, y, w, h = a.T
            gt[uid] = (c.astype(int), np.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], 1))
        else:
            gt[uid] = (np.zeros(0, int), np.zeros((0, 4)))
    return gt

def predict(model, df, imgsz, bs=32):
    # Ultralytics liste kaynağını tek batch yapıyor → listeyi elle böl
    files, out, t0 = [str(DATA / f) for f in df.file], {}, time.time()
    for s in range(0, len(files), bs):
        for r in model.predict(source=files[s:s + bs], imgsz=imgsz, conf=0.05, verbose=False):
            b = r.boxes
            out[Path(r.path).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(),
                                      b.xyxyn.cpu().numpy(), r.orig_shape)
    print(f'  {len(out)} görüntü · {time.time() - t0:.0f} sn')
    return out

DF = {'val': val, 'test': test}
GT = {k: read_gt(d) for k, d in DF.items()}
PR = {}
for e in (FINAL, BASE):
    m = YOLO(str(RUNS / e / 'weights' / 'best.pt'))
    for k in ('val', 'test'):
        print(e, k); PR[e, k] = predict(m, DF[k], IMGSZ[e])
    del m; gc.collect(); torch.cuda.empty_cache()

## 3. Güven eşiğini val'de seç

Her sınıf ve her eşik için kare bazında yakalama ve yanlış alarm. Seçilen eşik: val'de yanlış alarm ≤ `FA_MAX` olan **en düşük** eşik (yakalamayı en yüksek tutan). Bu eşikler test'te değiştirilmez.

In [ ]:
def frame_level(e, k, conf):
    rows = []
    d = DF[k]
    for c in (0, 1):
        has_gt = np.array([(GT[k][u][0] == c).any() for u in d.uid])
        has_pr = np.array([((PR[e, k][u][0] == c) & (PR[e, k][u][1] >= conf[c] if isinstance(conf, dict) else PR[e, k][u][1] >= conf)).any()
                           for u in d.uid])
        for src in ['hepsi'] + sorted(d.source.unique()):
            m = np.ones(len(d), bool) if src == 'hepsi' else (d.source == src).to_numpy()
            pos, neg = m & has_gt, m & ~has_gt
            rows.append({'model': e, 'küme': k, 'sınıf': NAMES[c], 'kaynak': src,
                         'pozitif kare': int(pos.sum()), 'yakalama %': 100 * has_pr[pos].mean() if pos.any() else np.nan,
                         'boş kare': int(neg.sum()), 'yanlış alarm %': 100 * has_pr[neg].mean() if neg.any() else np.nan})
    return rows

sweep = pd.DataFrame([dict(r, conf=c) for c in CONFS for r in frame_level(FINAL, 'val', float(c))
                      if r['kaynak'] == 'hepsi'])
THR = {}
for c in (0, 1):
    s = sweep[sweep.sınıf == NAMES[c]]
    ok = s[s['yanlış alarm %'] <= FA_MAX]
    THR[c] = float(ok.conf.min()) if len(ok) else float(s.conf.max())
print('seçilen eşikler (val):', {NAMES[c]: v for c, v in THR.items()})

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for i, c in enumerate((0, 1)):
    s = sweep[sweep.sınıf == NAMES[c]]
    ax[i].plot(s.conf, s['yakalama %'], 'o-', label='yakalama %')
    ax[i].plot(s.conf, s['yanlış alarm %'] * 10, 's--', label='yanlış alarm % ×10')
    ax[i].axvline(THR[c], color='k', ls=':', label=f'seçilen {THR[c]:.2f}')
    ax[i].set_title(f'{NAMES[c]} — val'); ax[i].set_xlabel('güven eşiği'); ax[i].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUTD / 'val_threshold.png', dpi=100); plt.show()
display(sweep.pivot_table(index='conf', columns='sınıf', values=['yakalama %', 'yanlış alarm %']).round(1))

## 4. Test — standart ölçüler (mAP)

Ultralytics'in kendi değerlendirmesi: tüm test, FASDD test ve Boreal test (Heinola) ayrı ayrı. Heinola'da sadece duman etiketi var.

In [ ]:
def ul_eval(e, yaml_name):
    m = YOLO(str(RUNS / e / 'weights' / 'best.pt'))
    r = m.val(data=str(DATA / yaml_name), split='test', imgsz=IMGSZ[e], batch=16, workers=WORKERS,
              plots=True, verbose=False, project=str(OUTD), name=f'{e}_{yaml_name.split(".")[0]}', exist_ok=True)
    rows = [{'model': e, 'test kümesi': yaml_name.split('.')[0].replace('data_test_', '').replace('data', 'hepsi'),
             'sınıf': 'all', 'P': r.box.mp, 'R': r.box.mr, 'mAP50': r.box.map50, 'mAP50-95': r.box.map}]
    for j, c in enumerate(r.box.ap_class_index):
        rows.append({'model': e, 'test kümesi': rows[0]['test kümesi'], 'sınıf': NAMES[int(c)],
                     'P': r.box.p[j], 'R': r.box.r[j], 'mAP50': r.box.ap50[j], 'mAP50-95': r.box.ap[j]})
    del m; gc.collect(); torch.cuda.empty_cache()
    return rows

ul = pd.DataFrame([row for e in (FINAL, BASE) for y in ('data.yaml', 'data_test_fasdd.yaml', 'data_test_boreal.yaml')
                   for row in ul_eval(e, y)]).round(3)
display(ul.pivot_table(index=['test kümesi', 'sınıf'], columns='model', values=['mAP50', 'mAP50-95', 'R'], sort=False))

## 5. Test — pratik ölçüler

Kare bazında yakalama / yanlış alarm, hem val'de seçilen eşikle hem de karşılaştırma için sabit 0.25 ile. Ayrıca sahne ortalaması (her sahne eşit ağırlıkta) ve piksel kapsama.

In [ ]:
fl = pd.DataFrame(frame_level(FINAL, 'test', THR) + frame_level(BASE, 'test', THR)
                  + [dict(r, model=r['model'] + ' @0.25') for e in (FINAL, BASE) for r in frame_level(e, 'test', 0.25)]).round(1)
display(fl.pivot_table(index=['sınıf', 'kaynak'], columns='model', values=['yakalama %', 'yanlış alarm %'], sort=False))

GRID = 128
def raster(boxes):
    m = np.zeros((GRID, GRID), bool)
    for x1, y1, x2, y2 in np.clip(boxes, 0, 1):
        m[int(y1 * GRID):max(int(np.ceil(y2 * GRID)), int(y1 * GRID) + 1),
          int(x1 * GRID):max(int(np.ceil(x2 * GRID)), int(x1 * GRID) + 1)] = True
    return m

def extra(e, k='test'):
    d = DF[k]; rows = []
    for c in (0, 1):
        px = np.zeros(3); hit = []
        for u, g in zip(d.uid, d.group):
            gc_, gb = GT[k][u]; pc, pf, pb, _ = PR[e, k][u]
            G, P = raster(gb[gc_ == c]), raster(pb[(pc == c) & (pf >= THR[c])])
            px += [G.sum(), P.sum(), (G & P).sum()]
            if (gc_ == c).any():
                hit.append((g, bool(P.any())))
        h = pd.DataFrame(hit, columns=['group', 'hit'])
        rows.append({'model': e, 'sınıf': NAMES[c], 'sahne sayısı': h.group.nunique(),
                     'sahne ort. yakalama %': 100 * h.groupby('group').hit.mean().mean() if len(h) else np.nan,
                     'piksel R': px[2] / max(px[0], 1), 'piksel P': px[2] / max(px[1], 1)})
    return rows

ex = pd.DataFrame(extra(FINAL) + extra(BASE)).round(3)
display(ex)

## 6. Hata örnekleri (test, final model, seçilen eşik)

Yeşil: etiket · kırmızı: tahmin. Üç grid: kaçırılan ateş kareleri, yanlış alarmlar (etiketsiz karede tahmin), Heinola'da kaçırılan duman.

In [ ]:
def show(uids, title, fname, cls=None):
    if not len(uids):
        print('örnek yok:', title); return
    n = len(uids)
    fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(12, 3.6 * ((n + 1) // 2)), squeeze=False)
    tf = test.set_index('uid').file
    for ax, u in zip(axes.ravel(), uids):
        with Image.open(DATA / tf[u]) as im:
            W, H = im.size; ax.imshow(im.convert('RGB'))
        gc_, gb = GT['test'][u]; pc, pf, pb, _ = PR[FINAL, 'test'][u]
        for c, col in ((0, 'lime'), (1, 'cyan')):
            if cls is not None and c != cls:
                continue
            for b in gb[gc_ == c]:
                ax.add_patch(patches.Rectangle((b[0] * W, b[1] * H), (b[2] - b[0]) * W, (b[3] - b[1]) * H, fill=False, ec=col, lw=1.2))
            for b, f in zip(pb[(pc == c) & (pf >= THR[c])], pf[(pc == c) & (pf >= THR[c])]):
                ax.add_patch(patches.Rectangle((b[0] * W, b[1] * H), (b[2] - b[0]) * W, (b[3] - b[1]) * H, fill=False, ec='red', lw=1.2, ls='--'))
                ax.text(b[0] * W, b[1] * H, f'{NAMES[c]} {f:.2f}', color='red', fontsize=6)
        ax.set_title(u[:40], fontsize=7); ax.axis('off')
    for ax in axes.ravel()[n:]:
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(OUTD / fname, dpi=90); plt.show()

def flags(c):
    has_gt = np.array([(GT['test'][u][0] == c).any() for u in test.uid])
    has_pr = np.array([((PR[FINAL, 'test'][u][0] == c) & (PR[FINAL, 'test'][u][1] >= THR[c])).any() for u in test.uid])
    return has_gt, has_pr

rng = np.random.default_rng(SEED)
fg, fp = flags(0); sg, sp = flags(1)
miss_fire = test.uid[fg & ~fp].to_numpy()
fa = test.uid[(~fg & fp) | (~sg & sp)].to_numpy()
miss_hein = test.uid[sg & ~sp & (test.source == 'boreal').to_numpy()].to_numpy()
print(f'kaçırılan ateş karesi {len(miss_fire)} · yanlış alarm karesi {len(fa)} · Heinola kaçırılan duman {len(miss_hein)}')
show(rng.permutation(miss_fire)[:6], 'Test — kaçırılan ateş', 'test_miss_fire.png', cls=0)
show(rng.permutation(fa)[:6], 'Test — yanlış alarm', 'test_false_alarm.png')
show(rng.permutation(miss_hein)[:6], 'Test — Heinola, kaçırılan duman', 'test_miss_heinola.png', cls=1)

## 7. Rapor

In [ ]:
def pick(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items():
        m &= (df[k] == v).to_numpy()
    return df[m]

def ulv(e, kume, sinif, col):
    r = pick(ul, model=e, **{'test kümesi': kume, 'sınıf': sinif})
    return f'{float(r[col].iloc[0]):.3f}' if len(r) else '—'

def flv(e, sinif, kaynak, col):
    r = pick(fl, model=e, sınıf=sinif, kaynak=kaynak)
    return f'%{float(r[col].iloc[0]):.1f}' if len(r) and pd.notna(r[col].iloc[0]) else '—'

L = ['# Final değerlendirme — test seti', '',
     f'- Final model: `{FINAL}` (val\'e göre seçildi) · baseline: `{BASE}`',
     f'- Güven eşikleri val\'de seçildi (kare bazında yanlış alarm ≤ %{FA_MAX}): fire {THR[0]:.2f}, smoke {THR[1]:.2f}',
     f'- Test: {len(test)} kare · FASDD {int((test.source == "fasdd").sum())} · Heinola (görülmemiş yangın) {int((test.source == "boreal").sum())}', '',
     '## Standart ölçüler (mAP)', '',
     '| test kümesi | sınıf | final mAP50 | final mAP50-95 | baseline mAP50 | baseline mAP50-95 |', '|---|---|---|---|---|---|']
for kume in ('hepsi', 'fasdd', 'boreal'):
    for s in ('all', 'fire', 'smoke'):
        if ulv(FINAL, kume, s, 'mAP50') != '—':
            L.append(f'| {kume} | {s} | {ulv(FINAL, kume, s, "mAP50")} | {ulv(FINAL, kume, s, "mAP50-95")} | '
                     f'{ulv(BASE, kume, s, "mAP50")} | {ulv(BASE, kume, s, "mAP50-95")} |')
L += ['', '## Kare bazında (val\'de seçilen eşikle)', '',
      '| sınıf | kaynak | final yakalama | final yanlış alarm | baseline yakalama | baseline yanlış alarm |', '|---|---|---|---|---|---|']
for s in ('fire', 'smoke'):
    for src in ('hepsi', 'fasdd', 'boreal'):
        if flv(FINAL, s, src, 'yakalama %') != '—' or flv(FINAL, s, src, 'yanlış alarm %') != '—':
            L.append(f'| {s} | {src} | {flv(FINAL, s, src, "yakalama %")} | {flv(FINAL, s, src, "yanlış alarm %")} | '
                     f'{flv(BASE, s, src, "yakalama %")} | {flv(BASE, s, src, "yanlış alarm %")} |')
L += ['', '## Sahne ortalaması ve piksel kapsama', '', ex.to_markdown(index=False), '',
      f'Kaçırılan ateş karesi: {len(miss_fire)} · yanlış alarm karesi: {len(fa)} · Heinola\'da kaçırılan duman karesi: {len(miss_hein)}']
txt = '\n'.join(L)
(REP / 'final_eval.md').write_text(txt)
json.dump({'final': FINAL, 'base': BASE, 'thr': {NAMES[c]: v for c, v in THR.items()},
           'ul': ul.to_dict('records'), 'frame': fl.to_dict('records'), 'extra': ex.to_dict('records')},
          open(OUTD / 'final_eval.json', 'w'), ensure_ascii=False, indent=1, default=float)
print(txt)

**Bana gönderilecekler:** son hücrenin çıktısı (`reports/final_eval.md` olarak da kaydedilir) ve üç hata grid'i hakkında birer cümle (kaçırılan ateşler nasıl: küçük/uzak/dumanın arkasında? yanlış alarmlar neye çıkmış: güneş, bulut, ışık?).

Sonra Faz 7: ONNX export + demo + README.